# Этап 1: поиск и аппроксимация пиков

Подгонка идёт по **несглаженному** спектру. Фон под пиками подгоняется прямой вместе с пиками.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ramancore.io import read_spectrum
from ramancore.preprocess import crop, remove_spikes
from ramancore.peaks import curves_on_grid, find_peaks, fit_peaks

## 1. Читаем спектр и ищем пики

In [ ]:
my_file = "../data/examples/synthetic_si.txt"  # ← подставьте свой файл
x, y, meta = read_spectrum(my_file)
x, y = crop(x, y, 100, 3200)
y = remove_spikes(y)

found = find_peaks(x, y)   # можно ослабить: find_peaks(x, y, min_snr=5)
print("Найдены пики около, см⁻¹:", np.round(found, 1))

## 2. Подгонка

Впишите в `centers` пики, которые хотите подогнать **вместе** (близкие, перекрывающиеся — например, TO и LO). Далёкие друг от друга пики лучше подгонять отдельными запусками.

`shape`: `"lorentz"`, `"gauss"` или `"voigt"` (псевдо-Фойгт).

In [ ]:
centers = found[:1]        # ← например: centers = [268, 292]
shape = "lorentz"
margin = 30                # сколько см⁻¹ брать по краям

table, fit = fit_peaks(x, y, centers, shape=shape, margin=margin)
print(f"R² = {fit['r2']:.4f}")
table.round(3)

In [ ]:
# График: точки, модель, отдельные пики, фон; внизу — остатки (данные − модель)
xf = np.linspace(fit["x"][0], fit["x"][-1], 1000)   # густая сетка для гладких кривых
baseline, components, total = curves_on_grid(table, fit, xf)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25], vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=fit["x"], y=fit["y"], mode="markers", name="данные"), row=1, col=1)
for comp, c in zip(components, table.center):   # zip — идём по двум спискам параллельно
    fig.add_trace(go.Scatter(x=xf, y=comp + baseline, name=f"пик {c:.1f}", line=dict(dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=xf, y=total, name="модель"), row=1, col=1)
fig.add_trace(go.Scatter(x=xf, y=baseline, name="фон", line=dict(dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=fit["x"], y=fit["y"] - fit["model"], mode="markers", name="остатки"), row=2, col=1)
fig.update_xaxes(title_text="Рамановский сдвиг, см⁻¹", row=2, col=1)
fig.update_yaxes(title_text="Интенсивность", row=1, col=1)
fig.show()

Как читать результат:
- `center ± center_err`, `fwhm ± fwhm_err`, `area ± area_err` — значения и погрешности 1σ;
- остатки должны быть похожи на шум без «волн»; волна под пиком значит, что форма пика выбрана неудачно — попробуйте `voigt`;
- FWHM включает аппаратное уширение спектрометра.